# Module 4 Mini Project: Customer Segmentation for a Retail Brand
**Technique:** K-means clustering on Recency, Frequency and Monetary (RFM) features

## Project objective
Group customers into four meaningful segments and recommend a targeted marketing strategy for each group.

**Dataset:** `OnlineRetail.csv` (Online Retail transactions). Place the CSV file in the same folder as this notebook.

### RFM meaning
- **Recency:** Number of days since the customer's most recent purchase. Lower is better.
- **Frequency:** Number of distinct invoices/orders. Higher usually means more repeat purchases.
- **Monetary:** Total sales value for the customer. Higher means greater historical spend.

We remove rows without a customer ID, cancelled invoices, non-positive quantities and non-positive prices before calculating RFM.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# 2. Load the dataset
# Keep OnlineRetail.csv in the same folder as this notebook.
df = pd.read_csv("OnlineRetail.csv", encoding="ISO-8859-1")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values:")
print(df.isna().sum())

In [ ]:
# 3. Clean the transaction data
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

data = df.dropna(subset=["CustomerID", "InvoiceDate"]).copy()
# Invoice numbers beginning with C indicate cancelled transactions.
data = data[~data["InvoiceNo"].astype(str).str.startswith("C")]
data = data[(data["Quantity"] > 0) & (data["UnitPrice"] > 0)]

data["CustomerID"] = data["CustomerID"].astype(int)
data["Sales"] = data["Quantity"] * data["UnitPrice"]

print("Rows after cleaning:", len(data))
print("Unique customers:", data["CustomerID"].nunique())
print("Latest transaction date:", data["InvoiceDate"].max())

In [ ]:
# 4. Create RFM features
# Snapshot date is one day after the latest transaction in the dataset.
snapshot_date = data["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = data.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda dates: (snapshot_date - dates.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Sales", "sum")
).reset_index()

print("Customers in RFM table:", len(rfm))
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

### Why transform and scale the data?
Monetary values and order counts are usually skewed. `log1p` reduces extreme skew, while `StandardScaler` puts the three features on a comparable scale. K-means uses distances, so scaling is important.

In [ ]:
# 5. Prepare features and fit K-means
features = ["Recency", "Frequency", "Monetary"]

# log(1 + x) safely handles zero values and reduces skew.
X_log = np.log1p(rfm[features])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# We choose 4 clusters to create four actionable customer segments.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

silhouette = silhouette_score(X_scaled, rfm["Cluster"])
print(f"Silhouette score: {silhouette:.3f}")
print("A silhouette score closer to 1 generally indicates better-separated clusters; interpret it alongside business usefulness.")

In [ ]:
# 6. Summarise the customer segments
segment_summary = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Median_Recency=("Recency", "median"),
    Median_Frequency=("Frequency", "median"),
    Median_Monetary=("Monetary", "median"),
    Total_Revenue=("Monetary", "sum")
).reset_index()

segment_summary["Customer_Share_%"] = (
    100 * segment_summary["Customers"] / len(rfm)
).round(1)
segment_summary["Revenue_Share_%"] = (
    100 * segment_summary["Total_Revenue"] / rfm["Monetary"].sum()
).round(1)

display(segment_summary.round(2))

In [ ]:
# 7. Visualise the segments
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

segment_summary.plot(
    x="Cluster", y="Customers", kind="bar", ax=axes[0], legend=False,
    title="Customers in each cluster"
)
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Number of customers")

segment_summary.plot(
    x="Cluster", y="Total_Revenue", kind="bar", ax=axes[1], legend=False,
    title="Total revenue by cluster"
)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Revenue")

plt.tight_layout()
plt.show()

## 8. Interpret the clusters and build a targeting strategy
Cluster numbers are arbitrary labels; they do **not** mean rank. Use the median RFM values to interpret each cluster. With this run, the segments are approximately:

| Cluster | Typical profile | Suggested action |
|---|---|---|
| 0 | Long time since last purchase, low order frequency and low spend | **At-risk / dormant:** win-back email, product reminders, limited-time offer; suppress if unresponsive |
| 1 | Recent purchase but low order frequency and spend | **New / occasional:** welcome journey, product education, cross-sell recommendations |
| 2 | Very recent, frequent purchases and high spend | **Champions / VIP:** loyalty benefits, early access, referral rewards, avoid unnecessary discounts |
| 3 | Moderate recency, repeat orders and medium-to-high spend | **Loyal / growth potential:** bundles, replenishment reminders, tailored cross-sell |

The segment labels are business interpretations, not labels created by the algorithm. Validate them against the table above before presenting the results.

In [ ]:
# 9. Save the customer-level segment assignments and summary
rfm.to_csv("customer_segments.csv", index=False)
segment_summary.to_csv("segment_summary.csv", index=False)

print("Saved: customer_segments.csv and segment_summary.csv")

## 10. Conclusion
K-means turns transaction history into customer groups that marketing teams can act on. Prioritise the high-value, active group for retention and advocacy; use personalised cross-sell for regular customers; and test low-cost win-back campaigns for dormant customers.

**Important limitations:** this is historical transaction data; the Monetary feature is total observed sales, not profit. The chosen number of clusters (four) should be compared with alternatives and reviewed with business stakeholders. The dataset ends in 2011, so these findings are a learning exercise and should not be treated as a current customer profile.